# Technical Demo: WikiRate/Wikidata Company Enrichment

This notebook is the runnable technical demo for the optional company-level enrichment layer.

The main Ethical Product Analyzer score still comes from product-level data: Open Food Facts, Open Beauty Facts, Open Products Facts, and the label-based scoring logic. WikiRate is not a replacement for that scoring engine. It is only an optional enrichment source for Social, Governance, and Ethics.

This notebook shows the full workflow: brand cleaning, Wikidata company resolution, WikiRate lookup, metric extraction, value classification, company-level adjustments, warnings, and updated score previews.

## Why Wikidata Is Needed

Open*Facts usually provides product brands, not parent companies. For example, a product may list `Nutella`, while company-level ESG data is more likely to exist under `Ferrero SpA`.

WikiRate generally stores company profiles and company-level ESG metrics. Therefore, the demo uses Wikidata as a bridge from brand to parent or owner company using:

- `P127` = owned by
- `P749` = parent organization

Example resolutions:

- Nutella -> Ferrero SpA
- Milka -> Mondelez International
- Nivea -> Beiersdorf
- KitKat -> Nestle
- Magnum -> Unilever

Low-confidence or ambiguous company resolution leads to no WikiRate adjustment.

In [48]:
from pathlib import Path
import importlib
import sys

import pandas as pd

PROJECT_ROOT_CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/Users/khadija/Desktop/WBS Coding School/ESG Project"),
]

PROJECT_ROOT = next(path for path in PROJECT_ROOT_CANDIDATES if (path / "wikirate_lookup.py").exists())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import wikirate_lookup
importlib.reload(wikirate_lookup)
wikirate_lookup.clear_caches()

from wikirate_lookup import (
    apply_company_adjustments,
    clean_brand_name,
    classify_metric_value,
    compute_company_adjustments,
    enrich_brand_with_company_esg,
    fetch_wikirate_metrics,
    get_first_brand,
    get_parent_company_from_wikidata,
    lookup_wikirate_company,
    resolve_brand_to_company,
    search_wikidata_entity,
)

print(f"Loaded wikirate_lookup from: {wikirate_lookup.__file__}")

Loaded wikirate_lookup from: /Users/khadija/Desktop/WBS Coding School/ESG Project/wikirate_lookup.py


## Step-by-Step Pipeline

1. Start with the Open*Facts `brands` field.
2. Use the first listed brand and clean the name.
3. Search Wikidata for the brand or company entity.
4. Resolve the parent or owner company through Wikidata `P127` / `P749` when available.
5. Query WikiRate for selected company-level ESG metrics.
6. Classify metric values as positive, negative, neutral, missing, or failed.
7. Compute small Social, Governance, and Ethics adjustments only when evidence is clear.
8. Return the enrichment separately as a preview for review.

Missing data, failed API requests, and low-confidence matches do not change the score.

The company enrichment confidence score measures how much usable company-level ESG evidence was found in WikiRate. It is separate from the product-level confidence score and does not change the score directly.

In [49]:
examples = ["Storck, Storck KG", "  Ben & Jerry's  ", "Milka"]

pd.DataFrame({
    "brands_field": examples,
    "first_brand": [get_first_brand(value) for value in examples],
    "clean_brand": [clean_brand_name(get_first_brand(value)) for value in examples],
})

,brands_field,first_brand,clean_brand
0,"Storck, Storck KG",Storck,storck
1,Ben & Jerry's,Ben & Jerry's,ben & jerry's
2,Milka,Milka,milka


## Metric Value Classification

The enrichment does not give points just because a metric exists. A positive policy metric means there is disclosure or policy evidence, not proof of perfect ethical behavior.

Examples of positive values include `Yes`, `True`, `Available`, `Disclosed`, or a published statement. Values such as `No`, `Unknown`, `Not available`, `No data`, failed requests, or HTTP 429 rate limits are not scored.

Negative adjustments are reserved for explicit controversy-style metrics. These are treated as controversy signals, not legal conclusions.

In [50]:
pd.DataFrame([
    {"metric": "Anti-corruption policy", "value": "Yes", "classification": classify_metric_value("Anti-corruption policy", "Yes")},
    {"metric": "Anti-corruption policy", "value": "No", "classification": classify_metric_value("Anti-corruption policy", "No")},
    {"metric": "Modern Slavery Statement", "value": "Yes - UK Modern Slavery Act", "classification": classify_metric_value("Modern Slavery Statement", "Yes - UK Modern Slavery Act")},
    {"metric": "Unknown policy", "value": "Not available", "classification": classify_metric_value("Unknown policy", "Not available")},
])

,metric,value,classification
0,Anti-corruption policy,Yes,positive
1,Anti-corruption policy,No,neutral
2,Modern Slavery Statement,Yes - UK Modern Slavery Act,positive
3,Unknown policy,Not available,unknown


## Test Brands

This section runs a clean test set from top to bottom. For each brand, the notebook shows company resolution, WikiRate lookup status, usable metrics, Social/Governance/Ethics adjustments, explanations, and warnings.

In [52]:
test_brands = [
    "Nutella",
    "Milka",
    "Nivea",
    "Ben & Jerry's",
    "Apple",
    "Dove",
    "KitKat",
    "Weleda",
    "Alnatura",
    "Storck",
]

enrichment_results = [enrich_brand_with_company_esg(brand) for brand in test_brands]

In [53]:
summary_rows = []

for result in enrichment_results:
    resolution = result["company_resolution"]
    wikirate = result["wikirate"]
    metrics = wikirate.get("metrics", {})
    usable_metrics_count = sum(
        metric.get("classification") in {"positive", "negative"}
        for metric in metrics.values()
    )
    adjustments = result["company_adjustments"]
    summary_rows.append({
        "brand": result["brand"],
        "resolved_company": resolution.get("parent_company_name"),
        "resolution_confidence": resolution.get("resolution_confidence"),
        "wikirate_company_found": wikirate.get("company_found"),
        "usable_metrics_count": usable_metrics_count,
        "company_enrichment_confidence": result["company_enrichment_confidence"],
        "company_enrichment_confidence_label": result["company_enrichment_confidence_label"],
        "social_adjustment": adjustments.get("social", 0),
        "governance_adjustment": adjustments.get("governance", 0),
        "ethics_adjustment": adjustments.get("ethics", 0),
        "warnings": result["warnings"],
    })

pd.DataFrame(summary_rows)


,brand,resolved_company,resolution_confidence,wikirate_company_found,usable_metrics_count,company_enrichment_confidence,company_enrichment_confidence_label,social_adjustment,governance_adjustment,ethics_adjustment,warnings
0,Nutella,Ferrero SpA,high,True,3,60,medium,0,9,3,[]
1,Milka,Mondelez International,high,True,9,100,high,10,10,-10,[Positive ethics policy bonuses suppressed bec...
2,Nivea,None,low,False,0,0,low,0,0,0,[Company could not be resolved confidently. No...
3,Ben & Jerry's,Ben & Jerry's,high,True,1,40,medium,4,0,2,[]
4,Apple,Apple Inc.,high,True,7,100,high,9,10,3,[]
5,Dove,None,low,False,0,0,low,0,0,0,[Company could not be resolved confidently. No...
6,KitKat,Nestlé,high,True,10,100,high,10,10,-10,[Positive ethics policy bonuses suppressed bec...
7,Weleda,Weleda,high,False,0,0,low,0,0,0,[Company not found on WikiRate. No company-lev...
8,Alnatura,Alnatura,high,True,0,20,low,0,0,0,[No usable metric found. Missing company data ...
9,Storck,None,low,False,0,0,low,0,0,0,[Company could not be resolved confidently. No...


## Inspect One Full Enrichment Result

Use this section to review the evidence behind a single brand. Every adjustment should be traceable to a metric name, value, year, classification, source or answer URL, affected pillar, and explanation.

In [54]:
enrich_brand_with_company_esg("Nutella")

{'brand': 'Nutella',
 'clean_brand': 'nutella',
 'base_scores': {'environmental': 70,
  'social': 55,
  'governance': 50,
  'ethics': 60},
 'company_resolution': {'clean_brand': 'nutella',
  'wikidata_qid': 'Q212193',
  'wikidata_label': 'Nutella',
  'wikidata_description': 'chocolate hazelnut spread manufactured by Ferrero',
  'parent_company_name': 'Ferrero SpA',
  'parent_company_qid': 'Q269792',
  'resolution_confidence': 'high',
  'resolution_method': 'wikidata_p127_or_p749',
  'reason': None},
 'wikirate': {'company_found': True,
  'company_name': 'Ferrero SpA',
  'company_url': 'https://wikirate.org/Ferrero_SpA.json?api_key=REDACTED',
  'metrics': {'modern_slavery_statement': {'metric_name': 'Business & Human Rights Resource Centre+Modern Slavery Statement',
    'value': 'Yes - UK Modern Slavery Act',
    'year': 2021,
    'classification': 'positive',
    'metric_kind': 'positive_policy',
    'pillar': 'ethics',
    'pillar_adjustments': {'ethics': 3, 'governance': 2},
    'adj

## Debug API Calls

Run with `debug=True` to inspect the API behavior. The debug output shows searched metric names, redacted API URLs, HTTP status codes, number of answers found, cache usage, and failed requests.

This is especially useful for identifying rate limiting or missing WikiRate coverage.

In [55]:
debug_result = enrich_brand_with_company_esg("Nutella", debug=True)
pd.DataFrame(debug_result["wikirate"].get("debug", []))

WikiRate company card URL: https://wikirate.org/Ferrero_SpA.json?api_key=REDACTED

Searched metric: human_rights_policy
  keywords: Human Rights Policy, Human Rights
  URL: https://wikirate.org/Ferrero_SpA+Answers.json?api_key=REDACTED&limit=20&filter%5Bmetric_keyword%5D=Human+Rights+Policy
  HTTP status: 200
  Answers returned: 0
  URL: https://wikirate.org/Ferrero_SpA+Answers.json?api_key=REDACTED&limit=20&filter%5Bmetric_keyword%5D=Human+Rights
  HTTP status: 200
  Answers returned: 0

Searched metric: modern_slavery_statement
  keywords: Modern Slavery Statement
  URL: https://wikirate.org/Ferrero_SpA+Answers.json?api_key=REDACTED&limit=20&filter%5Bmetric_keyword%5D=Modern+Slavery+Statement
  HTTP status: 200
  Answers returned: 6
  Selected metric card: Business & Human Rights Resource Centre+Modern Slavery Statement
  Selected year/value/classification: 2021 / Yes - UK Modern Slavery Act / positive

Searched metric: anti_corruption_policy
  keywords: Anti-Corruption Policy, Anti-

,metric_key,keyword,api_url,http_status,answers_returned,failed,reason,cached
0,human_rights_policy,Human Rights Policy,https://wikirate.org/Ferrero_SpA+Answers.json?...,200,0,False,None,True
1,human_rights_policy,Human Rights,https://wikirate.org/Ferrero_SpA+Answers.json?...,200,0,False,None,True
2,modern_slavery_statement,Modern Slavery Statement,https://wikirate.org/Ferrero_SpA+Answers.json?...,200,6,False,None,True
3,anti_corruption_policy,Anti-Corruption Policy,https://wikirate.org/Ferrero_SpA+Answers.json?...,200,0,False,None,True
4,anti_corruption_policy,Anti-bribery and anti-corruption,https://wikirate.org/Ferrero_SpA+Answers.json?...,200,0,False,None,True
5,anti_corruption_policy,corruption,https://wikirate.org/Ferrero_SpA+Answers.json?...,200,0,False,None,True
6,supply_chain_transparency,Supply Chain Transparency,https://wikirate.org/Ferrero_SpA+Answers.json?...,200,0,False,None,True
7,supply_chain_transparency,Traceability and Supply Chain Transparency,https://wikirate.org/Ferrero_SpA+Answers.json?...,200,0,False,None,True
8,supply_chain_transparency,Supply Chain,https://wikirate.org/Ferrero_SpA+Answers.json?...,200,4,False,None,True
9,worker_grievance_mechanism,Worker Grievance Mechanism,https://wikirate.org/Ferrero_SpA+Answers.json?...,200,0,False,None,True


## Updated Score Preview

This section demonstrates how company-level adjustments could be applied to sample base scores.

The preview does not modify the main product scoring engine. The enrichment remains separate so the evidence can be inspected before any future integration.

In [56]:
base_scores = {
    "environmental": 70,
    "social": 55,
    "governance": 50,
    "ethics": 60,
}

enrichment = enrich_brand_with_company_esg("Nutella", base_scores=base_scores)

{
    "base_scores": enrichment["base_scores"],
    "company_adjustments": enrichment["company_adjustments"],
    "updated_scores_preview": enrichment["updated_scores_preview"],
    "explanations": enrichment["explanations"],
    "warnings": enrichment["warnings"],
}


{'base_scores': {'environmental': 70,
  'social': 55,
  'governance': 50,
  'ethics': 60},
 'company_adjustments': {'social': 0, 'governance': 9, 'ethics': 3},
 'updated_scores_preview': {'environmental': 70,
  'social': 55,
  'governance': 59,
  'ethics': 63},
 'explanations': ['Business & Human Rights Resource Centre+Modern Slavery Statement found on WikiRate: Ethics +3',
  'Business & Human Rights Resource Centre+Modern Slavery Statement found on WikiRate: Governance +2',
  'Walk Free+MSA supply chain disclosure found on WikiRate: Governance +3',
  'Walk Free+MSA whistleblowing mechanism (binary) found on WikiRate: Governance +4'],
 'warnings': []}

## Optional Sample CSV Run

This section runs the same enrichment workflow on brands from the cleaned Open*Facts sample. It is useful for checking how many products can be resolved confidently and how often WikiRate has usable company-level evidence.

In [57]:
sample_path = PROJECT_ROOT / "workbooks" / "clean_openfacts_germany_sample.csv"
products = pd.read_csv(sample_path)

sample_brands = (
    products["brands"]
    .dropna()
    .astype(str)
    .map(get_first_brand)
    .map(str.strip)
    .loc[lambda values: values != ""]
    .drop_duplicates()
    .head(10)
    .tolist()
)

sample_brands

['Storck',
 'fin CARRE',
 'Fin carré',
 'Eat Natural',
 "Ben & Jerry's",
 'Milka',
 'MAGNUM',
 'Favorina',
 'dmBio',
 'Bon Gelati']

In [58]:
sample_results = [enrich_brand_with_company_esg(brand) for brand in sample_brands]

pd.DataFrame([
    {
        "brand": result["brand"],
        "company": result["company_resolution"].get("parent_company_name"),
        "confidence": result["company_resolution"].get("resolution_confidence"),
        "wikirate_found": result["wikirate"].get("company_found"),
        "usable_metrics": sum(
            metric.get("classification") in {"positive", "negative"}
            for metric in result["wikirate"].get("metrics", {}).values()
        ),
        "company_enrichment_confidence": result["company_enrichment_confidence"],
        "company_enrichment_confidence_label": result["company_enrichment_confidence_label"],
        "social_adjustment": result["company_adjustments"].get("social", 0),
        "governance_adjustment": result["company_adjustments"].get("governance", 0),
        "ethics_adjustment": result["company_adjustments"].get("ethics", 0),
        "warnings": result["warnings"],
    }
    for result in sample_results
])


,brand,company,confidence,wikirate_found,usable_metrics,company_enrichment_confidence,company_enrichment_confidence_label,social_adjustment,governance_adjustment,ethics_adjustment,warnings
0,Storck,None,low,False,0,0,low,0,0,0,[Company could not be resolved confidently. No...
1,fin CARRE,None,low,False,0,0,low,0,0,0,[Company could not be resolved confidently. No...
2,Fin carré,None,low,False,0,0,low,0,0,0,[Company could not be resolved confidently. No...
3,Eat Natural,Eat Natural,high,False,0,0,low,0,0,0,[Company not found on WikiRate. No company-lev...
4,Ben & Jerry's,Ben & Jerry's,high,True,1,40,medium,4,0,2,[]
5,Milka,Mondelez International,high,True,9,100,high,10,10,-10,[Positive ethics policy bonuses suppressed bec...
6,MAGNUM,Unilever,high,True,10,100,high,10,10,-10,[Positive ethics policy bonuses suppressed bec...
7,Favorina,Lidl,high,False,0,0,low,0,0,0,[Company not found on WikiRate. No company-lev...
8,dmBio,None,low,False,0,0,low,0,0,0,[Company could not be resolved confidently. No...
9,Bon Gelati,None,low,False,0,0,low,0,0,0,[Company could not be resolved confidently. No...
